In [1]:
#  TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [2]:
# TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

##  Implement Agent Logic Below

 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [3]:
#  AGENT FUNCTION (IMPLEMENTED)

import re

def _extract_math_expression(query: str) -> str:
    """Pull the math expression out of a query like 'Calculate 20 + 5'."""
    stripped = re.sub(r"calculate", "", query, flags=re.IGNORECASE).strip()
    match = re.search(r"[-+*/%().\d\s]+", stripped)
    expr = match.group(0).strip() if match else ""
    return expr


def _extract_keyword_text(query: str) -> str:
    """Pull the text payload out of a query like 'Extract keywords from ...'."""
    stripped = re.sub(r"keywords?", "", query, flags=re.IGNORECASE).strip()
    stripped = re.sub(r"^(extract|from|of|in|:|-)+\s*", "", stripped, flags=re.IGNORECASE).strip()
    return stripped if stripped else query


def agent(query: str):
    """
    Task-routing agent.

    Routes a natural-language query to the correct tool based on explicit
    keyword pattern checks on the lowercased query, and always returns a
    JSON-safe dict:
        {"type": "calculation" | "keywords" | "general" | "error", "result": ...}
    """
    # --- Guard: malformed / non-string input -----------------------------
    if not isinstance(query, str) or not query.strip():
        return {"type": "error", "result": "Invalid input: query must be a non-empty string"}

    query_lower = query.lower()

    # --- Route 1: calculation ---------------------------------------------
    if "calculate" in query_lower:
        try:
            expression = _extract_math_expression(query)
            if not expression:
                return {"type": "error", "result": "No valid mathematical expression found in query"}
            calc_result = calculator(expression)
            if calc_result == "Error in calculation":
                return {"type": "error", "result": calc_result}
            return {"type": "calculation", "result": calc_result}
        except Exception as e:
            return {"type": "error", "result": f"Calculation failed: {e}"}

    # --- Route 2: keyword extraction ---------------------------------------
    elif "keywords" in query_lower:
        try:
            payload = _extract_keyword_text(query)
            kws = extract_keywords(payload)
            return {"type": "keywords", "result": kws}
        except Exception as e:
            return {"type": "error", "result": f"Keyword extraction failed: {e}"}

    # --- Route 3: fallback general handler ----------------------------------
    else:
        try:
            response_text = (
                f"I received your query: \"{query.strip()}\". "
                "I can help with calculations (say 'calculate ...') "
                "or keyword extraction (say 'keywords ...')."
            )
            return {"type": "general", "result": response_text}
        except Exception as e:
            return {"type": "error", "result": f"Fallback handler failed: {e}"}

##  Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [4]:
#  Automated Validation: Test Cases

import json

queries = [
    "Calculate 20 + 5",                                             # valid calculation
    "Calculate (10 - 2) * 3",                                       # valid calculation
    "Extract keywords from Artificial Intelligence is transforming industries",  # valid keywords
    "keywords: The quick brown fox jumps over the lazy fox",        # valid keywords
    "What is machine learning?",                                    # general fallback
    "Hello, how are you today?",                                    # general fallback
    "Calculate 5 +",                                                # bad formatting -> error
    "Calculate import os",                                          # unsafe/invalid -> error
    "keywords",                                                     # no text supplied
    "",                                                             # empty string -> error
    12345,                                                          # non-string input -> error
]

REQUIRED_KEYS = {"type", "result"}
VALID_TYPES = {"calculation", "keywords", "general", "error"}

def validate_response(resp):
    """Check that a response is a JSON-valid dict with exactly type/result keys."""
    if not isinstance(resp, dict):
        return False, "response is not a dict"
    if set(resp.keys()) != REQUIRED_KEYS:
        return False, f"unexpected keys: {set(resp.keys())}"
    if resp["type"] not in VALID_TYPES:
        return False, f"unexpected type value: {resp['type']}"
    try:
        json.dumps(resp)
    except TypeError as e:
        return False, f"not JSON-serializable: {e}"
    return True, "ok"


print("=" * 70)
print("AUTOMATED VALIDATION RUN")
print("=" * 70)

pass_count = 0
for i, q in enumerate(queries, start=1):
    response = agent(q)
    is_valid, reason = validate_response(response)
    status = "PASS" if is_valid else "FAIL"
    pass_count += is_valid

    print(f"\n[{i:02d}] Query: {q!r}")
    print(f"     Validation: {status} ({reason})")
    print(f"     JSON output: {json.dumps(response)}")

print("\n" + "=" * 70)
print(f"SUMMARY: {pass_count}/{len(queries)} test cases produced valid JSON responses")
print("=" * 70)

assert pass_count == len(queries), "One or more validation checks failed!"
print("\nAll validation checks passed — no dropped or malformed responses.")

AUTOMATED VALIDATION RUN

[01] Query: 'Calculate 20 + 5'
     Validation: PASS (ok)
     JSON output: {"type": "calculation", "result": "25"}

[02] Query: 'Calculate (10 - 2) * 3'
     Validation: PASS (ok)
     JSON output: {"type": "calculation", "result": "24"}

[03] Query: 'Extract keywords from Artificial Intelligence is transforming industries'
     Validation: PASS (ok)
     JSON output: {"type": "keywords", "result": ["industries", "intelligence", "artificial", "transforming"]}

[04] Query: 'keywords: The quick brown fox jumps over the lazy fox'
     Validation: PASS (ok)
     JSON output: {"type": "keywords", "result": ["jumps", "quick", "brown"]}

[05] Query: 'What is machine learning?'
     Validation: PASS (ok)
     JSON output: {"type": "general", "result": "I received your query: \"What is machine learning?\". I can help with calculations (say 'calculate ...') or keyword extraction (say 'keywords ...')."}

[06] Query: 'Hello, how are you today?'
     Validation: PASS (ok)

In [5]:
#  Interactive Mode

print("Agent ready. Type 'exit' or 'quit' to stop.\n")
print("Try: 'Calculate 45 * (3 + 2)'  |  'keywords: Large language models use tools'  |  'tell me a joke'\n")

while True:
    try:
        user_input = input("Enter query (type 'exit' to stop): ")
    except EOFError:
        # Handles non-interactive kernels gracefully (e.g. automated notebook runs)
        print("No interactive input stream available — ending loop.")
        break

    if user_input.strip().lower() in {"exit", "quit"}:
        print("Agent: session ended.")
        break

    response = agent(user_input)
    is_valid, reason = validate_response(response)
    print("Response (JSON):", json.dumps(response))
    if not is_valid:
        print(f"  [validation warning: {reason}]")

Agent ready. Type 'exit' or 'quit' to stop.

Try: 'Calculate 45 * (3 + 2)'  |  'keywords: Large language models use tools'  |  'tell me a joke'

Response (JSON): {"type": "calculation", "result": "7"}
Response (JSON): {"type": "calculation", "result": "225"}
Response (JSON): {"type": "general", "result": "I received your query: \"stop\". I can help with calculations (say 'calculate ...') or keyword extraction (say 'keywords ...')."}
Agent: session ended.
